# DA6401 — System 2: Dropout 0.5, BatchNorm + Segmentation (frozen, partial)

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Quick Save** to run in background (~9 hrs)

**What this notebook does:**
- §2.1 BatchNorm ablation (no BN vs with BN)
- §2.2 Dropout experiment: p=0.5
- §2.3 Transfer learning: frozen + partial encoder freeze
- §2.6 Segmentation masks (auto-logged every 10 epochs)

**W&B tag:** `kaggle-sys2`

**Output files:**
- `checkpoints/classifier.pth` — best classifier (dropout=0.5 with BN)
- `checkpoints/classifier_nobn.pth` — no-batchnorm variant
- `checkpoints/classifier_drop05.pth` — dropout=0.5 with BN
- `checkpoints/unet.pth` — best segmentation model
- `checkpoints/unet_frozen.pth` — frozen encoder variant
- `checkpoints/unet_partial.pth` — partial freeze variant

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -2
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print(f"GPU: {gpu}")
assert torch.cuda.is_available(), "NO GPU — go to Settings and enable GPU T4 x2!"

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

## 1. Classification — BatchNorm + Dropout 0.5

**Important order:** BatchNorm ablation (no BN) runs FIRST so it doesn't overwrite the good classifier checkpoint. Then dropout=0.5 with BN runs and saves the final `classifier.pth` that segmentation will use.

In [ ]:
# ── §2.1  BatchNorm ablation: NO batchnorm (30 epochs to prove the point, not 60) ──
!python train.py --task classify --experiment kaggle-sys2 --no-bn --dropout 0.5 \
    --epochs 30 --lr 1e-4 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup the no-BN checkpoint separately
!cp {CKPT}/classifier.pth {CKPT}/classifier_nobn.pth
print("\n--- Backed up → classifier_nobn.pth (ablation proof) ---")

In [ ]:
# ── §2.2  Dropout = 0.5 WITH batchnorm (runs second → overwrites classifier.pth with good model) ──
!python train.py --task classify --experiment kaggle-sys2 --dropout 0.5 \
    --epochs 60 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup
!cp {CKPT}/classifier.pth {CKPT}/classifier_drop05.pth
print("\n--- Backed up → classifier_drop05.pth ---")
print("--- classifier.pth now has dropout=0.5 WITH BN → used as encoder for segmentation ---")

## 2. Segmentation — Transfer Learning (§2.3)

Uses the classifier's pretrained encoder from dropout=0.5 above.

Two freeze strategies in this notebook:
- **frozen**: entire encoder frozen, only decoder trains
- **partial**: blocks 0-2 frozen, blocks 3-4 + decoder train

In [ ]:
# ── §2.3  Segmentation: FROZEN encoder ──
!python train.py --task segment --experiment kaggle-sys2 --freeze-mode frozen \
    --epochs 60 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup
!cp {CKPT}/unet.pth {CKPT}/unet_frozen.pth
print("\n--- Backed up → unet_frozen.pth ---")

In [ ]:
# ── §2.3  Segmentation: PARTIAL freeze (blocks 0-2 frozen, 3-4 trainable) ──
!python train.py --task segment --experiment kaggle-sys2 --freeze-mode partial \
    --epochs 60 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup
!cp {CKPT}/unet.pth {CKPT}/unet_partial.pth
print("\n--- Backed up → unet_partial.pth ---")

## 3. Summary

In [ ]:
import os

CKPT = "/kaggle/working/checkpoints"

print("="*60)
print("  SYSTEM 2 — FINAL SUMMARY")
print("="*60)
print("\n  Files in /kaggle/working/checkpoints/:")
for f in sorted(os.listdir(CKPT)):
    size_mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"    {f:40s} {size_mb:6.1f} MB")

print("\n  W&B runs logged under tag: kaggle-sys2")
print("  Sections covered: §2.1 (BatchNorm), §2.2 (Dropout 0.5),")
print("                    §2.3 (Transfer: frozen + partial), §2.6 (Seg masks)")
print("\n  Download checkpoints from the Output tab after session ends!")
print("="*60)